# Simulação de dados de crédito

Este notebook cria uma base fictícia de crédito com PySpark.

In [ ]:
import os
import random
import sys
from datetime import datetime, timedelta

# Ensure Spark workers use this notebook's virtual environment on Windows.
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

from pyspark.sql import SparkSession
from pyspark.sql.functions import when

spark = (
    SparkSession.builder
    .appName("BaseFicticiaCredito")
    .getOrCreate()
)

## Funções auxiliares

In [ ]:
def gerar_cnpj_falso():
    nums = [random.randint(0, 9) for _ in range(14)]
    return (
        f"{nums[0]}{nums[1]}.{nums[2]}{nums[3]}{nums[4]}."
        f"{nums[5]}{nums[6]}{nums[7]}/0001-{nums[12]}{nums[13]}"
    )


def gerar_data_proxima(dias_max=30):
    return datetime.today().date() + timedelta(
        days=random.randint(1, dias_max)
    )

## Parâmetros e geração dos dados

In [ ]:
n_linhas = 10_000

status_opcoes = [
    "Aprovado Contratado",
    "Aprovado Não Contratado",
    "Rejeitado",
]
cnae_secoes = list("ABCDEFGHIJKLMNOPQRSTU")

dados = [
    (
        gerar_cnpj_falso(),
        gerar_data_proxima(),
        random.choice(status_opcoes),
        random.choice([0, 1]),
        random.choice(cnae_secoes),
    )
    for _ in range(n_linhas)
]

colunas = ["cnpj", "dt_ref", "status", "mau", "cnae_secao"]

## DataFrame Spark e status numérico

In [ ]:
df_spark = spark.createDataFrame(dados, colunas)

df_spark = df_spark.withColumn(
    "status_num",
    when(df_spark.status == "Aprovado Contratado", 2)
    .when(df_spark.status == "Aprovado Não Contratado", 1)
    .otherwise(0),
)

## Visualização e esquema

In [ ]:
df_spark.show(10, truncate=False)
df_spark.printSchema()

## Exportação opcional

Remova o comentário e ajuste o caminho para salvar os dados em Parquet.

In [ ]:
# df_spark.write.mode("overwrite").parquet("base_ficticia_credito")